In [ ]:
import hecfdapy as fda
import matplotlib.pyplot as plt


## The problem

Expected annual damage (EAD) is HEC-FDA's core risk metric: expected dollar damage per year, integrated over every flood event weighted by how often it happens. The compute chains four pieces together:

1. A **frequency curve** -- how often a flood of a given size occurs (an exceedance probability for each flow).
2. A **flow-stage** (rating) curve -- converts flow to a water-surface stage.
3. One or more **stage-damage curves** -- convert stage to dollar damage, per damage category and asset category.
4. An **integration** step -- combines damage and frequency across the whole curve into EAD.

Every point on every curve can carry sampling uncertainty (a distribution instead of a fixed number). `ead_simulation()` runs this as a seeded Monte Carlo: draw once from each distribution, walk the four steps, accumulate the result into a histogram, repeat, report the mean. With `deterministic=True` and a single iteration, every distribution collapses to its mean instead of sampling, so what's left is the plain arithmetic of the pipeline with no Monte Carlo noise at all.

This is the same construct as the getting-started example, spelled out properly this time: a Uniform(0, 100000) flow frequency, a flow-stage line from (0, 0) to (100000, 30), and a stage-damage curve that is flat at 600000 for any stage of 15 or more.


In [ ]:
sim = dict(
    flow_frequency=dict(dist="Uniform", params=[0, 100000, 1000]),
    flow_stage=dict(x=[0, 100000], dist="Uniform", params=[[0, 0, 10], [0, 30, 10]]),
    stage_damage=[dict(
        x=[0, 15, 30], dist="Uniform",
        params=[[0, 0, 10], [0, 600000, 10], [0, 600000, 10]],
        damage_category="residential", asset_category="Structure",
    )],
)

res = fda.ead_simulation(**sim, min_iterations=1, max_iterations=1, deterministic=True)
res["ead"], res["mean_aep"]


`res["ead"]` is 150000 -- the HEC-FDA deterministic oracle for this construct. `res["mean_aep"]` is the mean annual exceedance probability of the automatically-selected default performance threshold (HEC-FDA always derives one from the damage-frequency function itself, even with no threshold supplied): here it comes out to about 0.95, meaning this pipeline's own damage-producing range is exceeded in most years.


## A levee

A levee adds a **fragility curve**: the probability the levee fails, as a function of stage. Here the fragility is deterministic and binary -- 0% failure probability below stage 9.9999, 100% failure probability at stage 10 and above -- plus a `top_of_levee_elevation` marking where the levee itself sits.


In [ ]:
sim_levee = dict(sim, levee=dict(
    x=[0, 9.9999, 10], dist="Deterministic", params=[[0], [0], [1]],
    damage_category="residential", asset_category="Structure",
    top_of_levee_elevation=100000,
))

res_levee = fda.ead_simulation(**sim_levee, min_iterations=1, max_iterations=1, deterministic=True)
res_levee["ead"]


Adding the levee cuts expected annual damage from 150000 to 84187.50 -- HEC-FDA's own oracle for this exact construct. The levee doesn't erase damage: it only holds up to its own fragility curve, so above stage 10 damage still accrues at full risk. Below that, damage the base construct would have counted is now prevented.


## The seeded Monte Carlo

Switch `deterministic` off and every distribution samples instead of collapsing to its mean. Each curve draws from HEC-FDA's own seeded generator with a fixed constant per curve type -- frequency 1234, flow-stage 3456, stage-damage 6789, and four more for interior/exterior stage, system response, and life loss -- ported exactly from the C# source. The exact sequence of draws, and so the exact result, is reproducible across runs, across languages, and against the real HEC-FDA.

The fixture this example reproduces also tags the flow-stage curve with the same damage and asset category as the stage-damage curve; the seeded compute's random-number setup needs that tag to line up (the deterministic pass above works without it):


In [ ]:
sim = dict(sim, flow_stage=dict(sim["flow_stage"], damage_category="residential",
                                asset_category="Structure"))

res_100 = fda.ead_simulation(**sim, min_iterations=100, max_iterations=100, deterministic=False)
res_100["total_ead"]


At exactly 100 iterations this is `121194.5159789352` -- bit-for-bit the number the real HEC-FDA C# produces at the same iteration count, in both R and Python. That's the strongest check a Monte Carlo port can offer: not that two implementations land close together by chance, but that they draw the same sequence of random numbers and accumulate them the same way, all the way through the frequency -> stage -> damage -> integrate pipeline.


## Convergence

Run the same construct at increasing iteration counts and watch the seeded mean settle toward the deterministic value:


In [ ]:
iters = [100, 1000, 10000]
mean_ead = []
for n in iters:
    res_n = fda.ead_simulation(**sim, min_iterations=n, max_iterations=n, deterministic=False)
    mean_ead.append(res_n["total_ead"])

list(zip(iters, mean_ead))


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(iters, mean_ead, color="#588157", linewidth=2, marker="o", label="seeded Monte Carlo")
ax.axhline(150000, color="#cb997e", linestyle="--", linewidth=2, label="deterministic (150000)")
ax.set_xscale("log")
ax.set(xlabel="iterations", ylabel="mean EAD ($)")
ax.legend(frameon=False)


More iterations narrow the histogram's sampling noise around the same mean the deterministic pass computes directly. 150000 isn't a different answer -- it's the noise-free limit the seeded compute is estimating.


## Why the numbers are trustworthy

None of these values are made up. `150000`, `84187.50`, and `121194.5159789352` are the same literals the dotnet oracle gate (`make oracles`) checks against the real `HEC.FDA.Model.ImpactAreaScenarioSimulation.Compute`, and the last one is not just close: it is the exact value after exactly 100 iterations of the seeded Monte Carlo loop, reproduced bit-for-bit because the seven per-curve seed constants are ported exactly from the C# source, and every arithmetic step downstream of them matches too.
